### Imports

In [13]:
import warnings
import os
from model_function import *
from model_utils import *
from utils import *
from torch.utils.data import DataLoader
import torch.nn.functional as Fin
import timeit
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib 
from torchdiffeq import odeint as odeint
import matplotlib
matplotlib.use('Agg')
import argparse
import sys
import time
import torch
import torch.optim as optim
import random
import re
from collections import defaultdict

### Data loading

In [ ]:
levels = ["z","t","t2m","u10","v10"]
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
cwd = os.getcwd()
data_dir = ''
spectral = 0
scale = 0

train_time_scale= slice('2006','2016')
val_time_scale = slice('2016','2016')
test_time_scale = slice('2017','2018')

In [ ]:
paths_to_data = [str(data_dir) + '/era5_data/geopotential_500/*.nc',str(data_dir) + '/era5_data/temperature_850/*.nc',str(data_dir) + '/era5_data/2m_temperature/*.nc',str(data_dir) + '/era5_data/10m_u_component_of_wind/*.nc',str(data_dir) + '/era5_data/10m_v_component_of_wind/*.nc',str(data_dir) + '/era5_data/v_component_of_wind/*.nc',str(data_dir) + '/era5_data/u_component_of_wind/*.nc',str(data_dir) + '/era5_data/relative_humidity/*.nc',str(data_dir) + '/era5_data/specific_humidity/*.nc']
const_info_path = [str(data_dir) +  '/era5_data/constants/constants_5.625deg.nc']
levels = ["z","t","t2m","u10","v10"]
paths_to_data = paths_to_data[0:5]
levels = levels[0:5]
assert len(paths_to_data) == len(levels), "Paths to different type of data must be same as number of types of observations"
print("############################ Data is loading ###########################")
Final_train_data = 0
Final_val_data = 0
Final_test_data = 0
max_lev = []
min_lev = []
for idx,data in enumerate(paths_to_data):
    Train_data,Val_data,Test_data,time_steps,lat,lon,mean,std,time_stamp = get_train_test_data_without_scales_batched(data,train_time_scale,val_time_scale,test_time_scale,levels[idx],0)  
    max_lev.append(mean)
    min_lev.append(std)
    if idx==0: 
        Final_train_data = Train_data
        Final_val_data = Val_data
        Final_test_data = Test_data
    else:
        Final_train_data = torch.cat([Final_train_data,Train_data],dim=2)
        Final_val_data = torch.cat([Final_val_data,Val_data],dim=2)
        Final_test_data = torch.cat([Final_test_data,Test_data],dim=2)

print("Length of training data",len(Final_train_data))
print("Length of validation data",len(Final_val_data))
print("Length of testing data",len(Final_test_data))
const_channels_info,lat_map,lon_map = add_constant_info(const_info_path)

In [4]:
if spectral == 1: print("############## Running the Model in Spectral Domain ####################")
H,W = Train_data.shape[3],Train_data.shape[4]
clim = torch.mean(Final_test_data,dim=0)
Test_loader = DataLoader(Final_test_data[2:],batch_size=8,shuffle=False)
time_loader = DataLoader(time_steps[2:],batch_size=8,shuffle=False)
time_idx_steps = torch.tensor([i for i in range(365*4)]).view(-1,1)
time_idx = DataLoader(time_idx_steps[2:],batch_size=8,shuffle=False,pin_memory=False)

total_time_len = len(time_steps[2:])
total_time_steps = time_steps[2:].numpy().flatten().tolist()
num_years  = 2
Final_train_data = 0
Final_val_data = 0

### Ablation

In [5]:
def evaluate(model, vel_test, use_emission=True):
    R = {lev: [[] for _ in range(7)] for lev in levels}
    A = {lev: [[] for _ in range(7)] for lev in levels}
    C = {lev: [[] for _ in range(7)] for lev in levels}
    for entry, (time_steps, batch) in enumerate(zip(time_loader, Test_loader)):
        data = batch[0].to(device).view(num_years,1,len(paths_to_data)*(scale+1),H,W)
        past_sample = vel_test[entry].view(num_years,2*len(paths_to_data)*(scale+1),H,W).to(device)
        model.update_param([past_sample, const_channels_info.to(device),
                            lat_map.to(device), lon_map.to(device)])
        t = time_steps.float().to(device).flatten()
        with torch.no_grad():
            mean_pred, std_pred, mean_no_emission = model(t, data)
            if not use_emission:
                mean_pred = mean_no_emission

        for yr in range(2):
            for t_step in range(1, len(time_steps), 1):
                e_r = evaluation_rmsd_mm(mean_pred[t_step,yr].cpu(), batch[t_step,yr].cpu(),
                                         lat,lon,max_lev,min_lev,H,W,levels)
                e_a = evaluation_acc_mm(mean_pred[t_step,yr].cpu(), batch[t_step,yr].cpu(),
                                        lat,lon,max_lev,min_lev,H,W,levels,
                                        clim[yr].cpu().detach().numpy())
                if use_emission:
                    e_c = evaluation_crps_mm(mean_pred[t_step,yr].cpu(), batch[t_step,yr].cpu(),
                                            lat,lon,max_lev,min_lev,H,W,levels,
                                            std_pred[t_step,yr].cpu())
                for idx, lev in enumerate(levels):
                    R[lev][t_step-1].append(e_r[idx])
                    A[lev][t_step-1].append(e_a[idx])
                    if use_emission:
                        C[lev][t_step-1].append(e_c[idx])
    return {"RMSD": R, "ACC": A, "CRPS": C}

In [7]:
TEXTWIDTH_IN = 6.496  # 165 mm, \textwidth der scrreprt-Seite

plt.rcParams.update({
    # Vektorgrafik, keine Rasterisierung
    "figure.figsize": (TEXTWIDTH_IN, TEXTWIDTH_IN * 0.62),  # ~goldener Schnitt
    "savefig.format": "pdf",
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,

    # Schrift: STIX passt zu \rmdefault/\sfdefault = stix im Template
    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",

    # Schriftgrößen an 11pt-Fließtext / KOMA-Captions (~small) angleichen
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,

    # ruhiger, wissenschaftlicher Look statt matplotlib-Standard
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linewidth": 0.5,
    "lines.linewidth": 1.2,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "axes.linewidth": 0.8,
})

In [14]:
LEADS = [(i + 1) * 6 for i in range(7)]

TITLES = {
    "z":   "z500",
    "t":   "t850",
    "t2m": "t2m",
    "u10": "u10",
    "v10": "v10",
}
LABELS = {  # Namen wie in der Caption
    "ClimODE reference":  "ClimODE (checkpoint)",
    "ClimODE (Baseline)": "ClimODE (retrained)",
}

def plot_figure7(results, levels, metric="RMSD", metric_label="RMSE", fname=None):
    groups = defaultdict(list)
    for label, res in results.items():
        name = re.sub(r"\s*\(\d+\)$", "", label)
        name = re.sub(r"\s+\d+\)$", ")", name)
        groups[name].append(res)

    fig, axes = plt.subplots(2, 3, sharex=True,
                             figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * 0.55),
                             layout="constrained")
    axes = axes.ravel()

    for ax, lev in zip(axes, levels):
        for (label, runs), col in zip(groups.items(), plt.cm.tab10.colors):
            vals = np.array([[np.mean(r[metric][lev][i]) for i in range(7)] for r in runs])
            m = vals.mean(0)
            s = vals.std(0, ddof=1) if len(runs) > 1 else None
            ax.errorbar(LEADS, m, yerr=s, fmt="-o", ms=2.5, lw=1.0, capsize=4,
                        color=col, label=LABELS.get(label, label))
        ax.set_title(TITLES.get(lev, lev))
        ax.set_xticks(LEADS)

    # sharex blendet Ticklabels oben aus -> t2m (oben rechts) hat nichts darunter
    axes[2].tick_params(labelbottom=True)
    for ax in (axes[2], axes[3], axes[4]):
        ax.set_xlabel("Lead time [h]")
    for ax in (axes[0], axes[3]):
        ax.set_ylabel(f"latitude-weighted {metric_label}")

    # sechstes Feld = Legende
    axes[5].axis("off")
    handles, labels = axes[0].get_legend_handles_labels()
    axes[5].legend(handles, labels, loc="center", frameon=False)

    return fig

In [ ]:
ZERO_42 = "/checkpoints/ClimODE_global.pt"

CONFIGS = {
    "ClimODE": (ZERO_42, "None", []),
    "ClimODE no time": (ZERO_42, "None", ["t", "day", "seas", "pos_time"]),
    "ClimODE no nabla u": (ZERO_42, "None", ["nabla_u"]),
    "ClimODE no v": (ZERO_42, "None", ["v"]),
    "ClimODE no u": (ZERO_42, "None", ["u"]),
}

results = {}
for label, (ckpt, velfile, ablate) in CONFIGS.items():
    if velfile == "None":
        vel_test = torch.from_numpy(np.zeros((183, 2, 2, 5, 32, 64)))
    else:
        vel_test = torch.from_numpy(np.load(velfile))

    model = torch.load(
    str(cwd) + ckpt,
        map_location=torch.device('cpu'),
        weights_only=False,
    ).to(device)
    model.set_ablation(ablate)
    # model.eval()
    results[label] = evaluate(model, vel_test)
    print("fertig:", label)

In [ ]:
%matplotlib inline

plot_figure7(results, levels)
plt.savefig("zero_ablation.pdf")
plt.show()